# Custom Text Classification with BERT

Now, let’s fine-tune BERT on a real dataset.

Dataset: SMS Spam Classification https://github.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/blob/master/spam.csv

We will use the SMS Spam Dataset, where each SMS is labeled as:

* ham (not spam)

* spam (unwanted message)


# Step 1: Import Libraries

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Step 2: Load and Preprocess Dataset

In [4]:
import pandas as pd

# Load dataset using the raw URL and specify the encoding
df = pd.read_csv("https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv", encoding='ISO-8859-1')

# Rename columns
df.rename(columns={"v1": "label", "v2": "message"}, inplace=True)

# Convert labels to binary (0 for ham, 1 for spam)
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Drop unnecessary 'Unnamed' columns
df.drop(columns=[col for col in df.columns if 'Unnamed' in col], inplace=True)

# Check the data
print(df.head())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["message"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Step 3: Prepare Dataset for Training

In [7]:
class SpamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
train_dataset = SpamDataset(train_texts, train_labels)
val_dataset = SpamDataset(val_texts, val_labels)

# Step 4: Load Pretrained Model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Step 5: Define Metrics

In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Step 6: Training Arguments

In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Step 7: Train Model

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

  0%|          | 0/837 [00:00<?, ?it/s]

{'loss': 0.6456, 'grad_norm': 8.92874526977539, 'learning_rate': 5e-06, 'epoch': 0.04}
{'loss': 0.5091, 'grad_norm': 5.5113396644592285, 'learning_rate': 1e-05, 'epoch': 0.07}
{'loss': 0.3352, 'grad_norm': 3.3653736114501953, 'learning_rate': 1.5e-05, 'epoch': 0.11}
{'loss': 0.2005, 'grad_norm': 5.545851230621338, 'learning_rate': 2e-05, 'epoch': 0.14}
{'loss': 0.1312, 'grad_norm': 1.1573076248168945, 'learning_rate': 2.5e-05, 'epoch': 0.18}
{'loss': 0.067, 'grad_norm': 0.34145310521125793, 'learning_rate': 3e-05, 'epoch': 0.22}
{'loss': 0.0467, 'grad_norm': 0.3585726320743561, 'learning_rate': 3.5e-05, 'epoch': 0.25}
{'loss': 0.0034, 'grad_norm': 0.05859943479299545, 'learning_rate': 4e-05, 'epoch': 0.29}
{'loss': 0.1039, 'grad_norm': 41.13540267944336, 'learning_rate': 4.5e-05, 'epoch': 0.32}
{'loss': 0.1166, 'grad_norm': 19.83055877685547, 'learning_rate': 5e-05, 'epoch': 0.36}
{'loss': 0.0535, 'grad_norm': 0.07432612776756287, 'learning_rate': 4.932157394843962e-05, 'epoch': 0.39}


  0%|          | 0/18 [00:00<?, ?it/s]

{'eval_loss': 0.052816443145275116, 'eval_accuracy': 0.9901345291479821, 'eval_f1': 0.9619377162629758, 'eval_precision': 1.0, 'eval_recall': 0.9266666666666666, 'eval_runtime': 24.3746, 'eval_samples_per_second': 45.744, 'eval_steps_per_second': 0.738, 'epoch': 1.0}
{'loss': 0.0031, 'grad_norm': 0.20508547127246857, 'learning_rate': 3.778833107191316e-05, 'epoch': 1.0}
{'loss': 0.0798, 'grad_norm': 0.0487360917031765, 'learning_rate': 3.710990502035278e-05, 'epoch': 1.04}
{'loss': 0.0017, 'grad_norm': 0.04191510006785393, 'learning_rate': 3.643147896879241e-05, 'epoch': 1.08}
{'loss': 0.0012, 'grad_norm': 0.03464715927839279, 'learning_rate': 3.575305291723202e-05, 'epoch': 1.11}
{'loss': 0.0407, 'grad_norm': 0.02421165071427822, 'learning_rate': 3.5074626865671645e-05, 'epoch': 1.15}
{'loss': 0.0392, 'grad_norm': 0.03060356341302395, 'learning_rate': 3.4396200814111265e-05, 'epoch': 1.18}
{'loss': 0.1075, 'grad_norm': 0.06957758218050003, 'learning_rate': 3.3717774762550884e-05, 'epo

  0%|          | 0/18 [00:00<?, ?it/s]

{'eval_loss': 0.025698775425553322, 'eval_accuracy': 0.9946188340807175, 'eval_f1': 0.9798657718120806, 'eval_precision': 0.9864864864864865, 'eval_recall': 0.9733333333333334, 'eval_runtime': 28.1411, 'eval_samples_per_second': 39.622, 'eval_steps_per_second': 0.64, 'epoch': 2.0}
{'loss': 0.0293, 'grad_norm': 0.03240395337343216, 'learning_rate': 1.8792401628222527e-05, 'epoch': 2.01}
{'loss': 0.0236, 'grad_norm': 0.023288629949092865, 'learning_rate': 1.8113975576662146e-05, 'epoch': 2.04}
{'loss': 0.0008, 'grad_norm': 0.02024211548268795, 'learning_rate': 1.7435549525101765e-05, 'epoch': 2.08}
{'loss': 0.0417, 'grad_norm': 13.504704475402832, 'learning_rate': 1.6757123473541384e-05, 'epoch': 2.11}
{'loss': 0.0007, 'grad_norm': 0.0648694634437561, 'learning_rate': 1.6078697421981004e-05, 'epoch': 2.15}
{'loss': 0.0006, 'grad_norm': 0.016796818003058434, 'learning_rate': 1.5400271370420623e-05, 'epoch': 2.19}
{'loss': 0.0403, 'grad_norm': 0.01841980218887329, 'learning_rate': 1.472184

  0%|          | 0/18 [00:00<?, ?it/s]

{'eval_loss': 0.0226639024913311, 'eval_accuracy': 0.9955156950672646, 'eval_f1': 0.9832775919732442, 'eval_precision': 0.9865771812080537, 'eval_recall': 0.98, 'eval_runtime': 23.2372, 'eval_samples_per_second': 47.983, 'eval_steps_per_second': 0.775, 'epoch': 3.0}
{'train_runtime': 1108.8341, 'train_samples_per_second': 12.059, 'train_steps_per_second': 0.755, 'train_loss': 0.05512225711267562, 'epoch': 3.0}


TrainOutput(global_step=837, training_loss=0.05512225711267562, metrics={'train_runtime': 1108.8341, 'train_samples_per_second': 12.059, 'train_steps_per_second': 0.755, 'total_flos': 439757240152320.0, 'train_loss': 0.05512225711267562, 'epoch': 3.0})

# Step 8: Save Model

In [14]:
model.save_pretrained("./bert_spam_model")
tokenizer.save_pretrained("./bert_spam_model")

('./bert_spam_model/tokenizer_config.json',
 './bert_spam_model/special_tokens_map.json',
 './bert_spam_model/vocab.txt',
 './bert_spam_model/added_tokens.json',
 './bert_spam_model/tokenizer.json')

# Step 9: Load Model for Prediction

In [15]:
from transformers import AutoModelForSequenceClassification

In [16]:
model = AutoModelForSequenceClassification.from_pretrained("./bert_spam_model")
tokenizer = AutoTokenizer.from_pretrained("./bert_spam_model")

# Step 10: Predict Function

In [17]:
def predict(text):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Encode the text
    encoding = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    label = torch.argmax(probs, dim=1).item()
    confidence = probs[0][label].item()
    
    return ("Spam" if label == 1 else "Ham", confidence)

# Step 11: Test Predictions

In [18]:
messages = [
    "Congratulations! You've won a lottery. Claim now!",
    "Hey, are we meeting at 5?",
    "URGENT! Your account has been compromised. Click the link now to secure it.",
    "Don't forget the meeting tomorrow at 10am."
]

In [19]:
for msg in messages:
    print(f"Message: {msg} → Prediction: {predict(msg)}")

Message: Congratulations! You've won a lottery. Claim now! → Prediction: ('Ham', 0.8677036762237549)
Message: Hey, are we meeting at 5? → Prediction: ('Ham', 0.999767005443573)
Message: URGENT! Your account has been compromised. Click the link now to secure it. → Prediction: ('Ham', 0.506084680557251)
Message: Don't forget the meeting tomorrow at 10am. → Prediction: ('Ham', 0.9997327923774719)
